In [ ]:
import os
import numpy as np
from PIL import Image

def inspect_masks(mask_dir):

    print(f"Starting inspection of masks in directory: {mask_dir}")

    expected_pixel_values = {
        (0, 0, 0),
        (1, 1, 1),
        (2, 2, 2),
        (3, 3, 3),
        (4, 4, 4),
        (5, 5, 5)
    }
    

    problem_masks_found = 0
    
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])
    
    if not mask_files:
        print("No PNG files found in the masks directory.")
        return
    
    for filename in mask_files:
        mask_path = os.path.join(mask_dir, filename)
        
        try:
            mask_rgb = Image.open(mask_path).convert("RGB")
            mask_np = np.array(mask_rgb)
            
            unique_pixels = set(tuple(p) for p in mask_np.reshape(-1, mask_np.shape[-1]))
            
            unexpected_values = unique_pixels - expected_pixel_values
            
            if unexpected_values:
                problem_masks_found += 1
                print(f"--- Problematic mask found: {filename} ---")
                print(f"Unexpected pixel values found: {unexpected_values}")
                print(f"Expected values were: {expected_pixel_values}")
                
        except Exception as e:
            problem_masks_found += 1
            print(f"--- Error processing mask: {filename} ---")
            print(f"Error: {e}")
    
    print("\n--- Inspection Summary ---")
    if problem_masks_found == 0:
        print("All masks appear to be correct. No unexpected pixel values found.")
    else:
        print(f"Found {problem_masks_found} masks with issues. Please review the log.")

if __name__ == '__main__':
    mask_folder_path = 'C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/masks'
    inspect_masks(mask_folder_path)


In [ ]:

import cv2
import numpy as np
import os

input_dir  = "C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/masks"
output_dir = "C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/masks G"

os.makedirs(output_dir, exist_ok=True)

for fname in os.listdir(input_dir):
    if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    path = os.path.join(input_dir, fname)

    mask = cv2.imread(path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    out = np.zeros_like(mask, dtype=np.uint8)

    # color mapping
    out[np.all(mask == [0, 0, 0], axis=-1)] = [0, 0, 0]
    out[np.all(mask == [0, 255, 0], axis=-1)] = [1, 1, 1]
    out[np.all(mask == [255, 0, 0], axis=-1)] = [2, 2, 2]

    save_path = os.path.join(output_dir, fname)
    cv2.imwrite(save_path, cv2.cvtColor(out, cv2.COLOR_RGB2BGR))

print("✅ Bulk RGB mask conversion done.")


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(mask, cmap="tab10", vmin=0, vmax=2)
plt.colorbar()
plt.show()

In [ ]:
mask = cv2.imread(save_path, cv2.IMREAD_UNCHANGED)
print(np.unique(mask))

In [2]:

import cv2
import numpy as np
import os

input_dir  = r"C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/New"
output_dir = r"C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/New1"

os.makedirs(output_dir, exist_ok=True)

for fname in os.listdir(input_dir):
    if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    path = os.path.join(input_dir, fname)

    mask = cv2.imread(path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    out = np.zeros(mask.shape[:2], dtype=np.uint8)

    # class mapping
    out[np.all(mask == [0, 0, 0], axis=-1)] = 0  # black
    out[np.all(mask == [0, 255, 0], axis=-1)] = 1  # green
    out[np.all(mask == [255, 0, 0], axis=-1)] = 2  # red

    save_path = os.path.join(output_dir, fname)
    cv2.imwrite(save_path, out)

print("✅ Bulk conversion to single-channel class masks done.")


✅ Bulk conversion to single-channel class masks done.


In [ ]:

import os
import cv2
import numpy as np

input_dir = 'C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/tmp/masks RE'   
output_dir = 'C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/tmp/labels RE'  

classes = [
   
    ((255, 0, 0), 'weed'),
    ((0, 255, 0), 'Sunflower'),
    ((0, 0, 255), 'Background')]
   
os.makedirs(output_dir, exist_ok=True)

for image_name in os.listdir(input_dir):
    image_path = os.path.join(input_dir, image_name)
   
    mask = cv2.imread(image_path, cv2.IMREAD_COLOR)

    H, W, _ = mask.shape
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    
    mask_rgb_tuples = [tuple(pixel) for pixel in mask_rgb.reshape(-1, 3)]

    unique_rgb_tuples = list(set(mask_rgb_tuples))

    valid_rgb_tuples = [rgb_tuple for rgb_tuple in unique_rgb_tuples if rgb_tuple in [c[0] for c in classes]]

    polygons = []

    for rgb_tuple in valid_rgb_tuples:
        class_name = [c[1] for c in classes if c[0] == rgb_tuple][0]
        mask_rgb_indices = [i for i, x in enumerate(mask_rgb_tuples) if x == rgb_tuple]
        lower_bound = np.array([rgb_tuple[2], rgb_tuple[1], rgb_tuple[0]], dtype=np.uint8)
        upper_bound = np.array([rgb_tuple[2], rgb_tuple[1], rgb_tuple[0]], dtype=np.uint8)
        mask_binary = cv2.inRange(mask, lower_bound, upper_bound)
        mask_contour, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in mask_contour:
            if cv2.contourArea(contour) > 10:
                polygon = []
                for point in contour:
                    x, y = point[0]
                    polygon.append(x / W)
                    polygon.append(y / H)
                polygons.append((polygon, class_name))

    if polygons:

        output_file = os.path.join(output_dir, '{}.txt'.format(image_name[:-4]))
        with open(output_file, 'w') as f:
            for polygon, class_name in polygons:
                class_number = [c[1] for c in classes].index(class_name)
                f.write('{} '.format(class_number))
                for p in polygon:
                    f.write('{} '.format(p))
                f.write('\n')

    print('Processed:', image_name)

print('Process completed.')


In [ ]:
# Image Compression/Size reduction

from PIL import Image
import os

in_dir = "./images/"
out_dir = "./images_compressed/"
os.makedirs(out_dir, exist_ok=True)

for file in os.listdir(in_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        img = Image.open(os.path.join(in_dir, file))
        out_path = os.path.join(out_dir, file.replace(".png", ".jpg"))  
        img.save(out_path, "JPEG", quality=85)  


In [ ]:
# Creates train.txt and val.txt files for DeepLab training

    import os
    import random

    def create_deeplab_txt_files(image_folder, output_list_folder, train_ratio=0.9):
        """
        Creates train.txt and val.txt files for DeepLab training.

        Args:
            image_folder (str): Path to the folder containing your image files.
            output_list_folder (str): Path to the folder where train.txt and val.txt will be saved.
            train_ratio (float): The proportion of images to allocate to the training set.
        """
        if not os.path.exists(output_list_folder):
            os.makedirs(output_list_folder)

        image_filenames = [os.path.splitext(f)[0] for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
        random.shuffle(image_filenames)

        num_train = int(len(image_filenames) * train_ratio)
        train_filenames = image_filenames[:num_train]
        val_filenames = image_filenames[num_train:]

        with open(os.path.join(output_list_folder, 'train.txt'), 'w') as f:
            for filename in train_filenames:
                f.write(filename + '\n')

        with open(os.path.join(output_list_folder, 'val.txt'), 'w') as f:
            for filename in val_filenames:
                f.write(filename + '\n')

        print(f"Created train.txt with {len(train_filenames)} images and val.txt with {len(val_filenames)} images.")


    image_directory = "./images_compressed"
    list_directory = "./Segmentation"
    create_deeplab_txt_files(image_directory, list_directory)

In [3]:
# Converts a single YOLO segmentation label file (.txt) to a mask image.

import os
import numpy as np
from PIL import Image, ImageDraw

def yolo_seg_to_mask(txt_file, img_size, num_classes):
    
    h, w = img_size
    # Create an empty mask with a background value of 0.
    mask = np.zeros((h, w), dtype=np.int64)

    try:
        with open(txt_file, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: Label file not found at {txt_file}. Skipping.")
        return mask

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 3:
            continue  # Skip malformed lines

        try:
            cls = int(float(parts[0]))
            coords = list(map(float, parts[1:]))
        except (ValueError, IndexError):
            print(f"Warning: Skipping malformed line in {txt_file}: {line.strip()}")
            continue


        if len(coords) >= 2:
            poly = [(int(coords[i] * w), int(coords[i+1] * h)) for i in range(0, len(coords), 2)]
            

            img = Image.fromarray(np.zeros((h, w), dtype=np.uint8))
            ImageDraw.Draw(img).polygon(poly, fill=cls + 1)
            

            mask = np.maximum(mask, np.array(img, dtype=np.int64))

    return mask

def bulk_yolo_seg_to_masks(labels_dir, masks_dir, img_size, num_classes):

    if not os.path.exists(masks_dir):
        os.makedirs(masks_dir)
    
    label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
    
    if not label_files:
        print(f"No .txt files found in the labels directory: {labels_dir}")
        return

    print(f"Found {len(label_files)} label files to process.")
    
    for filename in label_files:
        txt_path = os.path.join(labels_dir, filename)
        
        mask_filename = os.path.splitext(filename)[0] + '.png'
        mask_path = os.path.join(masks_dir, mask_filename)
        
        mask = yolo_seg_to_mask(txt_path, img_size, num_classes)
        
        Image.fromarray(mask.astype(np.uint8)).save(mask_path)
        print(f"Converted '{filename}' to '{mask_filename}'")

if __name__ == "__main__":
    image_height, image_width = 512, 512
    num_classes = 5 
    
    labels_folder = 'C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/val/'
    masks_folder = 'C:/Users/PMLS/Downloads/Yolo New/yolov8/image-segmentation-yolov8-main-mine/WGSEG/val m/'
    
    if not os.path.exists(labels_folder):
        os.makedirs(labels_folder)
    with open(os.path.join(labels_folder, 'image1.txt'), 'w') as f:
        f.write("0 0.2 0.2 0.4 0.2 0.4 0.4 0.2 0.4\n")
        f.write("1 0.6 0.6 0.8 0.6 0.8 0.8 0.6 0.8\n")
    with open(os.path.join(labels_folder, 'image2.txt'), 'w') as f:
        f.write("2 0.1 0.1 0.3 0.1 0.2 0.3 0.1 0.3\n")
    
    bulk_yolo_seg_to_masks(labels_folder, masks_folder, (image_height, image_width), num_classes)
    
    print("\nBulk conversion complete. Check the 'masks' directory for results.")



Found 16 label files to process.
Converted '001-GRE.txt' to '001-GRE.png'
Converted '001-NIR.txt' to '001-NIR.png'
Converted '001-REG.txt' to '001-REG.png'
Converted '004-REG.txt' to '004-REG.png'
Converted '010-GRE.txt' to '010-GRE.png'
Converted '013-NIR.txt' to '013-NIR.png'
Converted '014-GRE.txt' to '014-GRE.png'
Converted '015-REG.txt' to '015-REG.png'
Converted '016-NIR.txt' to '016-NIR.png'
Converted '022-RED.txt' to '022-RED.png'
Converted '024-GRE.txt' to '024-GRE.png'
Converted '024-NIR.txt' to '024-NIR.png'
Converted '024-REG.txt' to '024-REG.png'
Converted '038-GRE.txt' to '038-GRE.png'
Converted 'image1.txt' to 'image1.png'
Converted 'image2.txt' to 'image2.png'

Bulk conversion complete. Check the 'masks' directory for results.
